# Neo4j를 활용한 뉴스 데이터 그래프 데이터베이스 구축


## 데이터 구조

네이버 뉴스 데이터를 그래프 데이터베이스에 저장하고 분석합니다.

**데이터 구성**: 
- 카테고리 (경제, 생활/문화, IT/과학, 세계)
- 뉴스 제목 및 내용
- 언론사
- 기자명
- 발행년월

## 1. 그래프 데이터베이스란?


### 관계형 DB vs 그래프 DB

**관계형 데이터베이스 (RDBMS)**:
- 테이블 형태로 데이터 저장
- JOIN으로 관계 표현 (성능 저하 가능)
- 복잡한 관계 분석이 어려움

**그래프 데이터베이스 (Neo4j)**:
- 노드(Node)와 관계(Relationship)로 데이터 저장
- 관계가 1급 객체로 저장됨
- 복잡한 관계 분석에 최적화


### Neo4j의 구성 요소

1. **노드(Node)**: 엔티티 (예: 뉴스, 기자, 언론사)
2. **관계(Relationship)**: 노드 간 연결 (예: WRITTEN_BY, PUBLISHED_BY)
3. **속성(Property)**: 노드와 관계의 데이터 (예: 제목, 날짜)
4. **레이블(Label)**: 노드의 타입 (예: :News, :Reporter)

## 2. 그래프 모델 설계

### 2.1 노드(Node) 설계

```
:News (뉴스 기사)
  - title: 뉴스_제목
  - link: 링크
  - content: 뉴스_내용
  - publishDate: 발행일자

:Category (카테고리)
  - name: 카테고리명 (예: "경제", "생활/문화")

:Publisher (언론사)
  - name: 언론사명 (예: "국민일보", "SBS")

:Reporter (기자)
  - name: 기자명 (예: "이광수 기자")

:YearMonth (발행년월)
  - period: "YYYY-MM" 형식 (예: "2025-11")
```

### 2.2 관계(Relationship) 설계

```
(:News)-[:BELONGS_TO]->(:Category)
  → 뉴스가 특정 카테고리에 속함

(:News)-[:PUBLISHED_BY]->(:Publisher)
  → 뉴스가 특정 언론사에 의해 발행됨

(:News)-[:WRITTEN_BY]->(:Reporter)
  → 뉴스가 특정 기자에 의해 작성됨

(:News)-[:PUBLISHED_IN]->(:YearMonth)
  → 뉴스가 특정 년월에 발행됨

(:Reporter)-[:WORKS_FOR]->(:Publisher)
  → 기자가 특정 언론사에 소속됨
```

### 2.3 그래프 구조 다이어그램

```
            ┌─────────────┐
            │  Category   │
            │   (경제)    │
            └─────────────┘
                  ↑
                  │ BELONGS_TO
                  │
        ┌─────────┴─────────┐
        │       News        │
        │   (뉴스 기사)      │
        └───────────────────┘
             │          │
             │          │ WRITTEN_BY
             │          ↓
             │     ┌──────────┐      WORKS_FOR    ┌──────────┐
             │     │ Reporter │─────────────────→ │Publisher │
             │     │ (이광수) │                   │(국민일보) │
             │     └──────────┘                   └──────────┘
             │                                          ↑
             │ PUBLISHED_BY                             │
             └──────────────────────────────────────────┘
                              │
                              │ PUBLISHED_IN
                              ↓
                        ┌──────────┐
                        │YearMonth │
                        │(2025-11) │
                        └──────────┘
```


## 3. Docker를 활용한 Neo4j 환경 구축

### 3.1 Docker Compose 파일 구성

프로젝트 루트에 `docker-compose.yml` 파일이 이미 준비되어 있습니다.

**주요 설정**:
- **포트**: 7474 (웹 브라우저), 7687 (Bolt 프로토콜)
- **인증정보**: ID: `neo4j`, PW: `test1234`
- **플러그인**: APOC, Graph Data Science
- **메모리**: 힙 메모리 512MB~2GB, 페이지 캐시 1GB


### 3.2 Neo4j 시작하기

터미널에서 다음 명령어를 실행하세요:

```shell
# Neo4j 컨테이너 시작
docker-compose up -d

# 컨테이너 상태 확인
docker ps

# 로그 확인 (Neo4j가 정상적으로 시작되었는지 확인)
docker-compose logs neo4j
```

### 3.3 Neo4j Browser 접속

1. 웹 브라우저를 열고 http://localhost:7474 접속
2. 로그인 정보 입력:
   - **Connect URL**: `bolt://localhost:7687`
   - **Username**: `neo4j`
   - **Password**: `test1234`
3. "Connect" 버튼 클릭

✅ 정상적으로 접속되면 준비 완료!


## 4. Python으로 Neo4j 연결하기

### 4.1 필요한 라이브러리 설치

requirements.txt에 있는 라이브러리들이 설치되어 있어야 합니다:
```shell
# 터미널에서 실행:
# pip install -r requirements.txt

# 또는 주피터 노트북에서 직접 설치:
%pip install pandas neo4j
```


### 4.2 라이브러리 임포트 및 데이터 로드


In [ ]:
import pandas as pd
from neo4j import GraphDatabase
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('data/naver_news_detail_20251128.csv')

# 데이터 확인
print(f"전체 데이터: {len(df)}개")
print(f"\n데이터 미리보기:")
df.head()


### 4.3 Neo4j 연결 클래스 생성


In [ ]:
class Singleton(type):
	_instances = {}

	def __call__(cls, *args, **kwargs):
		if cls not in cls._instances:
			cls._instances[cls] = super(Singleton, cls)\
				.__call__(*args, **kwargs)
		return cls._instances[cls]

In [ ]:
class Neo4jConnection(metaclass=Singleton):
    """Neo4j 데이터베이스 연결 및 작업을 위한 클래스"""
    
    def __init__(self, uri, user, password):
        """
        Neo4j 연결 초기화
        
        Args:
            uri: Neo4j 서버 주소 (예: "bolt://localhost:7687")
            user: 사용자 이름
            password: 비밀번호
        """
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("Neo4j 연결 성공!")
    
    def close(self):
        """연결 종료"""
        if self.driver is not None:
            self.driver.close()
            print("Neo4j 연결 종료")
    
    def execute_query(self, query, parameters=None):
        """
        Cypher 쿼리 실행
        
        Args:
            query: 실행할 Cypher 쿼리
            parameters: 쿼리 파라미터 (딕셔너리)
        
        Returns:
            쿼리 실행 결과
        """
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]


### 4.4 Neo4j 연결 생성

In [ ]:
# Neo4j 연결 생성
conn = Neo4jConnection(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="test1234"
)

### 4.5 기존 데이터 모두 삭제

In [ ]:
# 기존 데이터 모두 삭제
delete_query = """
MATCH (n)
DETACH DELETE n
"""

conn.execute_query(delete_query)
print("기존 데이터 삭제 완료")


## 5. Neo4j에 뉴스 데이터 저장하기

### 5.1 인덱스 생성 (성능 최적화)

데이터를 삽입하기 전에 인덱스를 생성하면 검색 성능이 크게 향상됩니다.


In [ ]:
# 인덱스 생성 쿼리들
index_queries = [
    "CREATE INDEX category_name IF NOT EXISTS FOR (c:Category) ON (c.name)",
    "CREATE INDEX publisher_name IF NOT EXISTS FOR (p:Publisher) ON (p.name)",
    "CREATE INDEX reporter_name IF NOT EXISTS FOR (r:Reporter) ON (r.name)",
    "CREATE INDEX yearmonth_period IF NOT EXISTS FOR (ym:YearMonth) ON (ym.period)",
    "CREATE INDEX news_title IF NOT EXISTS FOR (n:News) ON (n.title)"
]

print("인덱스 생성 중...")
for query in index_queries:
    conn.execute_query(query)
    
print("인덱스 생성 완료!")


### 5.2 뉴스 데이터 삽입

이제 CSV 데이터를 Neo4j 그래프로 변환하여 삽입합니다.


In [ ]:
def insert_news_to_neo4j(df, conn):
    """
    뉴스 데이터를 Neo4j에 삽입
    
    Args:
        df: 뉴스 데이터프레임
        conn: Neo4j 연결 객체
    """
    
    insert_query = """
    // 1. Category 노드 생성 (중복 방지: MERGE 사용)
    MERGE (cat:Category {name: $category})
    
    // 2. Publisher 노드 생성
    MERGE (pub:Publisher {name: $publisher})
    
    // 3. Reporter 노드 생성
    MERGE (rep:Reporter {name: $reporter})
    
    // 4. Reporter와 Publisher 관계 생성
    MERGE (rep)-[:WORKS_FOR]->(pub)
    
    // 5. YearMonth 노드 생성
    MERGE (ym:YearMonth {period: $yearMonth})
    
    // 6. News 노드 생성
    CREATE (news:News {
        title: $title,
        link: $link,
        content: $content,
        publishDate: $publishDate
    })
    
    // 7. 관계 생성
    CREATE (news)-[:BELONGS_TO]->(cat)
    CREATE (news)-[:PUBLISHED_BY]->(pub)
    CREATE (news)-[:WRITTEN_BY]->(rep)
    CREATE (news)-[:PUBLISHED_IN]->(ym)
    
    RETURN news.title as title
    """
    
    total = len(df)
    success_count = 0
    
    print(f"총 {total}개의 뉴스 데이터 삽입 시작...\n")
    
    for idx, row in df.iterrows():
        try:
            # 발행년월 추출 (YYYY-MM 형식)
            publish_date = row['발행일자']
            year_month = publish_date[:7]  # "2025-11-27 18:51:11" -> "2025-11"
            
            # 파라미터 준비
            parameters = {
                'category': row['카테고리'],
                'publisher': row['언론사'],
                'reporter': row['기자명'],
                'yearMonth': year_month,
                'title': row['뉴스_제목'],
                'link': row['링크'],
                'content': row['뉴스_내용'][:500] if pd.notna(row['뉴스_내용']) else "",  # 내용은 500자로 제한
                'publishDate': publish_date
            }
            
            # 쿼리 실행
            conn.execute_query(insert_query, parameters)
            success_count += 1
            
            # 진행상황 출력
            if (idx + 1) % 5 == 0:
                print(f"{idx + 1}/{total} 완료 ({success_count}개 성공)")
                
        except Exception as e:
            print(f"오류 발생 (행 {idx + 1}): {str(e)}")
    
    print(f"\n데이터 삽입 완료! 총 {success_count}/{total}개 성공")



In [ ]:
# 데이터 삽입 실행
insert_news_to_neo4j(df, conn)

### 5.3 저장된 뉴스 데이터 확인 

In [ ]:
# 노드 개수 확인
count_query = """
MATCH (n)
RETURN labels(n)[0] as NodeType, count(n) as Count
ORDER BY Count DESC
"""

results = conn.execute_query(count_query)

print("노드 통계:\n")
print(f"{'노드 타입':<15} {'개수':>10}")
print("-" * 27)
for record in results:
    print(f"{record['NodeType']:<15} {record['Count']:>10,}")
    

In [ ]:
# 관계 개수 확인
relationship_query = """
MATCH ()-[r]->()
RETURN type(r) as RelationType, count(r) as Count
ORDER BY Count DESC
"""

results = conn.execute_query(relationship_query)

print("\n관계 통계:\n")
print(f"{'관계 타입':<20} {'개수':>10}")
print("-" * 32)
for record in results:
    print(f"{record['RelationType']:<20} {record['Count']:>10,}")


## 6. Cypher 쿼리로 데이터 분석하기

Cypher는 Neo4j의 쿼리 언어입니다. SQL과 비슷하지만 그래프 구조에 최적화되어 있습니다.


### 6.1 기본 쿼리 예제

In [ ]:
# 예제 1: 모든 카테고리 조회
query1 = """
MATCH (c:Category)
RETURN c.name as 카테고리
"""

results = conn.execute_query(query1)
print("카테고리 목록:")
for record in results:
    print(f"  - {record['카테고리']}")


In [ ]:
# 예제 2: "경제" 카테고리의 뉴스 개수
query2 = """
MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: "경제"})
RETURN count(n) as 뉴스개수
"""

results = conn.execute_query(query2)
for record in results:
    print(f"경제 카테고리 뉴스: {record['뉴스개수']}개")


In [ ]:
# 예제 3: 각 카테고리별 뉴스 개수
query3 = """
MATCH (n:News)-[:BELONGS_TO]->(c:Category)
RETURN c.name as 카테고리, count(n) as 뉴스개수
ORDER BY 뉴스개수 DESC
"""

results = conn.execute_query(query3)
print("\n카테고리별 뉴스 통계:\n")
for record in results:
    print(f"  {record['카테고리']:<12} : {record['뉴스개수']:>3}개")


### 6.2 언론사 및 기자 분석


In [ ]:
# 예제 4: 가장 많은 기사를 작성한 언론사 Top 5
query4 = """
MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher)
RETURN p.name as 언론사, count(n) as 기사수
ORDER BY 기사수 DESC
LIMIT 5
"""

results = conn.execute_query(query4)
print("기사 발행 Top 5 언론사:\n")
for i, record in enumerate(results, 1):
    print(f"  {i}. {record['언론사']:<15} : {record['기사수']}개")


In [ ]:
# 예제 5: 특정 언론사의 기자 목록과 작성 기사 수
query5 = """
MATCH (r:Reporter)-[:WORKS_FOR]->(p:Publisher {name: "뉴스1"})
OPTIONAL MATCH (r)-[:WRITTEN_BY]-(n:News)
RETURN r.name as 기자명, count(n) as 작성기사수
ORDER BY 작성기사수 DESC
"""

results = conn.execute_query(query5)
print("\n뉴스1의 기자 목록:\n")
for record in results:
    print(f"  {record['기자명']:<15} : {record['작성기사수']}개")


### 6.3 복잡한 관계 쿼리


In [ ]:
# 예제 6: 특정 기자가 작성한 기사의 카테고리 분포
query6 = """
MATCH (r:Reporter)-[:WRITTEN_BY]-(n:News)-[:BELONGS_TO]->(c:Category)
WHERE r.name CONTAINS "기자"
WITH r.name as 기자명, c.name as 카테고리, count(n) as 기사수
RETURN 기자명, 카테고리, 기사수
ORDER BY 기자명, 기사수 DESC
LIMIT 10
"""

results = conn.execute_query(query6)
print("기자별 카테고리 작성 분포:\n")
current_reporter = None
for record in results:
    if current_reporter != record['기자명']:
        current_reporter = record['기자명']
        print(f"\n  {current_reporter}:")
    print(f"    - {record['카테고리']}: {record['기사수']}개")


In [ ]:
# 예제 7: 2025년 11월에 발행된 뉴스의 언론사별 통계
query7 = """
MATCH (n:News)-[:PUBLISHED_IN]->(ym:YearMonth {period: "2025-11"})
MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
RETURN p.name as 언론사, count(n) as 기사수
ORDER BY 기사수 DESC
"""

results = conn.execute_query(query7)
print("\n2025년 11월 언론사별 기사 발행:\n")
for record in results:
    print(f"  {record['언론사']:<15} : {record['기사수']}개")


In [ ]:
# 예제 8: 특정 키워드가 포함된 뉴스 검색
query8 = """
MATCH (n:News)-[:BELONGS_TO]->(c:Category)
MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
WHERE n.title CONTAINS "AI" OR n.title CONTAINS "인공지능"
RETURN n.title as 제목, c.name as 카테고리, p.name as 언론사
LIMIT 5
"""

results = conn.execute_query(query8)
print("\n'AI/인공지능' 관련 뉴스:\n")
for i, record in enumerate(results, 1):
    print(f"  {i}. [{record['카테고리']}] {record['제목']}")
    print(f"     출처: {record['언론사']}\n")


### 6.4 고급 쿼리: 경로(Path) 찾기


In [ ]:
# 예제 9: 카테고리 -> 뉴스 -> 기자 -> 언론사 경로
query9 = """
MATCH path = (c:Category {name: "IT/과학"})<-[:BELONGS_TO]-(n:News)-[:WRITTEN_BY]->(r:Reporter)-[:WORKS_FOR]->(p:Publisher)
RETURN c.name as 카테고리, n.title as 뉴스제목, r.name as 기자, p.name as 언론사
LIMIT 3
"""

results = conn.execute_query(query9)
print("IT/과학 카테고리 뉴스의 경로 추적:\n")
for i, record in enumerate(results, 1):
    print(f"{i}. 경로:")
    print(f"   [{record['카테고리']}] → [{record['뉴스제목'][:30]}...] → [{record['기자']}] → [{record['언론사']}]\n")


## 7. 정리 및 리소스 해제

In [ ]:
# 기존 데이터 모두 삭제
delete_query = """
MATCH (n)
DETACH DELETE n
"""

conn.execute_query(delete_query)
print("기존 데이터 삭제 완료")

In [ ]:
# 연결 종료
conn.close()
print("Neo4j 연결 종료!")